# 03 - Scaling & Normalization
Transform numeric features into comparable scales for clustering, classification, and forecasting models.
Strategies: StandardScaler (z-score), MinMaxScaler ([0,1]), RobustScaler (median & IQR).
Input: `data/processed/02_*.csv` | Output: `data/processed/03_scaled_*.csv` with original + scaled columns.

In [ ]:
import numpy as np, pandas as pd, warnings
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
PROJECT_ROOT = Path.cwd().parent.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
datasets = {}
for fname in ['02_worldbank_features.csv', '02_wto_demand_features.csv', '02_diversification_metrics.csv']:
    path = PROCESSED_DIR / fname
    if path.exists():
        datasets[fname.replace('.csv', '')] = pd.read_csv(path)
        print(f'Loaded {fname}: {datasets[fname.replace(".csv", "")].shape}')
    else:
        print(f'Skipped {fname} (not found)')

In [ ]:
def get_numeric_cols(df, exclude=None):
    if exclude is None:
        exclude = ['year', 'refyear', 'country_code', 'reporteriso', 'partneriso', 'cmdcode', 'product_sector_code', 'indicator', 'source', 'flow_type']
    nums = df.select_dtypes(include=[np.number]).columns.tolist()
    return [c for c in nums if c not in exclude and not c.endswith(('_std', '_minmax', '_robust'))]
def scale_dataframe(df, numeric_cols, scaler, suffix):
    df_out = df.copy()
    available = [c for c in numeric_cols if c in df_out.columns and df_out[c].notna().any()]
    if not available:
        return df_out
    fill_vals = df_out[available].median()
    scaled_matrix = scaler.fit_transform(df_out[available].fillna(fill_vals))
    for i, col in enumerate(available):
        df_out[f'{col}_{suffix}'] = scaled_matrix[:, i]
    return df_out
scalers = {'std': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}
print('Scalers ready:', list(scalers.keys()))

## 3.1 - Scale World Bank Macro Features

In [ ]:
key = '02_worldbank_features'
if key in datasets:
    df = datasets[key].copy()
    num_cols = get_numeric_cols(df)
    print('World Bank numeric columns:', num_cols)
    for suffix, scaler in scalers.items():
        df = scale_dataframe(df, num_cols, scaler, suffix)
    datasets[key] = df
    print('Scaled World Bank shape:', df.shape)
    scaled_cols = [c for c in df.columns if c.endswith(('_std', '_minmax', '_robust'))]
    display(df[['country', 'year'] + scaled_cols[:6]].head())
else:
    print('World Bank dataset not available.')

## 3.2 - Scale WTO Demand Features

In [ ]:
key = '02_wto_demand_features'
if key in datasets and not datasets[key].empty:
    df = datasets[key].copy()
    num_cols = get_numeric_cols(df)
    print('WTO numeric columns:', num_cols)
    for suffix, scaler in scalers.items():
        df = scale_dataframe(df, num_cols, scaler, suffix)
    datasets[key] = df
    print('Scaled WTO demand shape:', df.shape)
    scaled_cols = [c for c in df.columns if c.endswith(('_std', '_minmax', '_robust'))]
    display(df[['year', 'product_sector'] + scaled_cols[:6]].head())
else:
    print('WTO demand dataset not available or empty.')

## 3.3 - Scale Diversification Metrics

In [ ]:
key = '02_diversification_metrics'
if key in datasets and not datasets[key].empty:
    df = datasets[key].copy()
    num_cols = get_numeric_cols(df)
    print(f'{key} numeric columns:', num_cols)
    for suffix, scaler in scalers.items():
        df = scale_dataframe(df, num_cols, scaler, suffix)
    datasets[key] = df
    print('Scaled shape:', df.shape)
else:
    print(f'{key} not available or empty.')

## 3.4 - Save All Scaled Datasets

In [ ]:
output_mapping = {
    '02_worldbank_features': '03_worldbank_scaled.csv',
    '02_wto_demand_features': '03_wto_demand_scaled.csv',
    '02_diversification_metrics': '03_diversification_scaled.csv'
}
for key, out_name in output_mapping.items():
    if key in datasets and not datasets[key].empty:
        out_path = PROCESSED_DIR / out_name
        datasets[key].to_csv(out_path, index=False)
        print(f'Saved {out_name} -> shape {datasets[key].shape}')
    else:
        print(f'Skipped {out_name} (source missing)')
print('\nScaling complete. Proceed to 04_master_integration_and_eda.ipynb.')